# PG-RAD to PyMC pipeline -- Activity difference

This notebook is an automation pipeline to sample two-source geometries of varying source activities.

The script first generates scenarios using PG-RAD with source activity difference as a variable.

The generated data is passed into the Bayesian pipeline, and 10 simulations are done. The traces are written to disk and can be analyzed in the `trace-analysis.ipynb` notebook.


## Imports

In [12]:
TRACE_OUT_FOLDER = "trace_output/"

In [13]:
from pathlib import Path
import json
import time
import sys

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import yaml
import arviz as az

from matplotlib import pyplot as plt
import matplotlib.patches as patches

from pg_rad.utils.projection import rel_to_abs_source_position as convert_pos

# src directory
parent_dir = Path.cwd().parent.parent 
sys.path.append(str(parent_dir))

print(parent_dir)

from tools import *
from run_model import run

/home/pim/pg-rad-analysis/src


## Helper function

To load the simulations for Bayesian inference

In [14]:
def get_sim_path_and_csv_name(i, source_separation_FWHM, base_dir="output"):
    base_path = Path(base_dir)
    # Find all directories matching the pattern f"{i}_result_*"
    folders = list(base_path.glob(f"{i}_fwhm_{source_separation_FWHM}_act_diff_result_*"))
    if not folders:
        raise FileNotFoundError(f"No folder matching {i}_{source_separation_FWHM}_result_* found in {base_dir}")

    folder_name = folders[0].name
    print("Detected folder name:", folder_name)

    # Find all CSV files in the folder
    csv_files = list(folders[0].glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folders[0]}")

    csv_name = csv_files[0].name
    print("Detected CSV name:", csv_name)

    return str(folders[0]), csv_name

## Loading the base configuration

Below the base config for source separation scenario is loaded. The parameters regarding source geometry will be modified dynamically using the JSON library of Python.

In [15]:
with open('base_config.yml') as f:
    conf = yaml.safe_load(f)
conf

{'name': 'test',
 'speed': 8.33,
 'acquisition_time': 1,
 'path': {'length': [100, 1000], 'segments': [{'turn_left': 30}, 'straight']},
 'sources': {'s1': {'activity_MBq': 300,
   'isotope': 'Cs137',
   'gamma_energy_keV': 662,
   'position': [0, 0, 0]},
  's2': {'activity_MBq': 300,
   'isotope': 'Cs137',
   'gamma_energy_keV': 662,
   'position': [0, 0, 0]}},
 'detector': 'LU_HPGe_90',
 'options': {'bkg_cps': 0}}

## Simulation parameters

The `DETECTOR` and `DET_EFF` parameters will be used for the Bayesian part, and need to be set correctly.

In [16]:
DETECTOR = "HPGe"
DET_EFF = 0.002410082036721662

Parameters here can be changed if desired, e.g. the source activity or the distance to the road

In [17]:
FWHM = 5.52 * conf['speed'] * conf['acquisition_time'] # calculated externally, FWHM for source at 32 meters from the road.

In [22]:
source_separation_FWHM = 2 # multiples of FWHM
r_1 = 32
r_2 = r_1

A_1 = 300
A_2_ratios = [1., 1.5, 2., 2.5, 3.]

We create a DataFrame containing all the scenarios

In [23]:
df = pd.DataFrame(columns=["d", "A1", "A2", "r1", "r2", "A1/A2", "r1/r2"])

df['A1'] = np.full_like(A_2_ratios, A_1)
df['A2']= A_1 / np.array(A_2_ratios)
df['d'] = np.full_like(df['A1'], source_separation_FWHM)

df['A1/A2'] = df['A1'] / df['A2'] 
df['r1'] = np.full_like(df['A1'], r_1)
df['r2'] = np.full_like(df['A1'], r_2)
df['r1/r2'] = df['r1'] / df['r2'] 
df.head()

,d,A1,A2,r1,r2,A1/A2,r1/r2
0,2.0,300.0,300.0,32.0,32.0,1.0,1.0
1,2.0,300.0,200.0,32.0,32.0,1.5,1.0
2,2.0,300.0,150.0,32.0,32.0,2.0,1.0
3,2.0,300.0,120.0,32.0,32.0,2.5,1.0
4,2.0,300.0,100.0,32.0,32.0,3.0,1.0


## Generating the scenarios using PG-RAD

Below is a for loop which will iterate over the rows of the above DataFrame and its configurations, dynamically write a new PG-RAD configuration, generate the scenario and save the output.

In [24]:
conf_list = []

for index, row in df.iterrows():
    d = row['d'] * FWHM
    conf['sources']['s1']['activity_MBq'] = float(row['A1'])
    conf['sources']['s2']['activity_MBq'] = float(row['A2'])
    conf['sources']['s1']['position'] = {'along_path': float(700 - d/2), 'dist_from_path': float(row['r1']), 'side': 'left'}
    conf['sources']['s2']['position'] = {'along_path': float(700 + d/2), 'dist_from_path': float(row['r2']), 'side': 'left'}
    conf['name'] = str(round(row['A1/A2'], 2))+f'_fwhm_{source_separation_FWHM}_act_diff'

    with open('tmp.yml', 'w') as f:
        yaml.dump(conf, f, default_flow_style=False)
    
    ! cd pgrad_output && pgrad --config ../tmp.yml --save

2026-05-15 17:07:28,136 - INFO: Landscape built successfully: 1.0_fwhm_2_act_diff
<class 'numpy.bool'>
2026-05-15 17:07:28,185 - INFO: Simulation output saved to 1.0_fwhm_2_act_diff_result_20260515_1707!
2026-05-15 17:07:31,677 - INFO: Landscape built successfully: 1.5_fwhm_2_act_diff
<class 'numpy.bool'>
2026-05-15 17:07:31,712 - INFO: Simulation output saved to 1.5_fwhm_2_act_diff_result_20260515_1707!
2026-05-15 17:07:35,199 - INFO: Landscape built successfully: 2.0_fwhm_2_act_diff
<class 'numpy.bool'>
2026-05-15 17:07:35,232 - INFO: Simulation output saved to 2.0_fwhm_2_act_diff_result_20260515_1707!
2026-05-15 17:07:38,855 - INFO: Landscape built successfully: 2.5_fwhm_2_act_diff
<class 'numpy.bool'>
2026-05-15 17:07:38,905 - INFO: Simulation output saved to 2.5_fwhm_2_act_diff_result_20260515_1707!
2026-05-15 17:07:42,431 - INFO: Landscape built successfully: 3.0_fwhm_2_act_diff
<class 'numpy.bool'>
2026-05-15 17:07:42,465 - INFO: Simulation output saved to 3.0_fwhm_2_act_diff_re

## Bayesian inference

In [26]:
for i in df['A1/A2']:
    i = round(i, 2)
    csv_path, roi_filename = get_sim_path_and_csv_name(i, source_separation_FWHM, base_dir='pgrad_output')
    BASE_DIR = Path("__file__").resolve().parent
    CSV_DIR = Path(BASE_DIR.joinpath(csv_path))

    params_file = Path(CSV_DIR.joinpath("parameters.json"))
    params = json.load(params_file.open())

    csv_file = Path(CSV_DIR.joinpath(roi_filename))
    df_i = pd.read_csv(csv_file)

    df_i['ROI_BR'] += 15
    df_i['ROI_P'] += df_i['ROI_BR']

    for j in range(10): # 10 to compute localization prob.
        trace, real_params = run(df_i, params, csv_file, DETECTOR, DET_EFF)
        az.to_netcdf(trace, filename=TRACE_OUT_FOLDER+str(i)+f"_fwhm_{source_separation_FWHM}_trace_"+str(j))
        df_i.to_pickle(TRACE_OUT_FOLDER+str(i)+f"_fwhm_{source_separation_FWHM}_pkl_"+str(j))
    
        with open(TRACE_OUT_FOLDER+str(i)+f"_fwhm_{source_separation_FWHM}_real_params_"+str(j),'w') as fp:
            json.dump(real_params, fp)

Detected folder name: 1.0_fwhm_2_act_diff_result_20260515_1707
Detected CSV name: 1_2_src_0_cps_bkg_300MBq_300MBq_32m_32m_567_332_647_378.csv


/home/pim/miniforge3/envs/pg-rad-analysis/lib/python3.12/site-packages/pymc/sampling/mcmc.py:786: UserWarning: A list or tuple of random_seed no longer specifies the specific random_seed of each chain. Use a single seed instead.
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [y_src, x_src, act_src]


Output()

Sampling 2 chains for 500 tune and 2_000 draw iterations (1_000 + 4_000 draws total) took 18 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Sampling: [predicted_cps]


Output()

...

[!] No 2 peaks found. Continuing...


/home/pim/miniforge3/envs/pg-rad-analysis/lib/python3.12/site-packages/pymc/sampling/mcmc.py:786: UserWarning: A list or tuple of random_seed no longer specifies the specific random_seed of each chain. Use a single seed instead.
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [y_src, x_src, act_src]


Output()

Sampling 2 chains for 500 tune and 2_000 draw iterations (1_000 + 4_000 draws total) took 32 seconds.
There were 37 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [predicted_cps]


Output()